In [ ]:
import pandas as pd
import os

# 경로 설정 ( 혹은 탐색 )
paths_to_try = [
    r"C:\Users\JG\Downloads"
]

folder_path = ""
for p in paths_to_try:
    if os.path.exists(p):
        folder_path = p
        break

# 만약 위 경로 중 맞는 게 없다면, 현재 이 파일이 있는 폴더를 주소로 잡습니다.
if not folder_path:
    folder_path = os.path.dirname(os.path.abspath(__file__))

print(f"[진단] 현재 파이썬이 조사 중인 폴더: {folder_path}")
try:
    print(f"[진단] 폴더 안의 실제 파일 목록: {os.listdir(folder_path)}")
except Exception as e:
    print(f"폴더를 열 수 없음: {e}")


# 2. CSV와 Excel 파일을 대소문자 구분 없이 모두 찾습니다.
all_files = []
for f in os.listdir(folder_path):
    ext = os.path.splitext(f)[-1].lower()
    if ext in ('.csv', '.xlsx'):
        full_path = os.path.join(folder_path, f)
        if "cheongna_clean_base" not in f and f.startswith("검단_당하동"):
            all_files.append(full_path)

print(f"📊 발견된 데이터 파일 총 {len(all_files)}개")

all_shards = []

# 3. 파일별 확장자(CSV vs Excel)에 맞춰 자동으로 다르게 읽어오기
for file in all_files:
    ext = os.path.splitext(file)[-1].lower()
    file_name = os.path.basename(file)
    
    try:
        if ext == '.csv':
            print(f"📄 CSV 파일 읽는 중: {file_name}")
            try:
                df = pd.read_csv(file, skiprows=15, encoding='cp949', dtype={'번지': str})
            except UnicodeDecodeError:
                df = pd.read_csv(file, skiprows=15, encoding='utf-8-sig', dtype={'번지': str})
        else:
            print(f"📊 Excel 파일 읽는 중: {file_name}")
            df = pd.read_excel(file, skiprows=16)
            
        if not df.empty:
            all_shards.append(df)
    except Exception as e:
        print(f"❌ 파일 읽기 실패 ({file_name}): {e}")

# 4. 파일이 하나도 없을 때 에러로 튕기는 것 방지
if not all_shards:
    print("\n❌ [경고] 합칠 파일이 여전히 0개입니다. 데이터 파일들이 올바른 폴더에 있는지 확인해주세요.")
else:
    # 하나로 병합
    raw_df = pd.concat(all_shards, ignore_index=True)
    print(f"🎉 합쳐진 초기 데이터 건수: {len(raw_df)}개")

    # ✅ 전용면적 숫자 변환 (문자열 → float)
    raw_df['전용면적(㎡)'] = pd.to_numeric(raw_df['전용면적(㎡)'], errors='coerce')

    print(f"① 전체: {len(raw_df)}개")

    # ✅ 맨 앞에 '도시명' 컬럼 추가, 값은 모두 '검단'
    raw_df.insert(0, '도시명', '검단')
    raw_df = raw_df.drop(columns=['NO'])

    # 번지 이상값 확인용 (임시)
    mask = raw_df['번지'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$|^[A-Za-z]{3}-\d{2}$')
    print(raw_df[mask]['번지'].unique())

    # ✅ 번지가 날짜 형식(1983-01-01 또는 Jan-83 등)인 행 제거
    before = len(raw_df)
    raw_df = raw_df[~raw_df['번지'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$|^[A-Za-z]{3}-\d{2}$')]
    print(f"② 번지 이상값 제거: {before - len(raw_df)}개 제거 → {len(raw_df)}개 남음")

    # 결과 저장
    output_path = os.path.join(folder_path, "geomdan_dangha_merged.csv")
    raw_df.to_csv(output_path, index=False, encoding='cp949')
    print(f"💾 최종 파일 저장 완료: {output_path}")

In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\2471369\Downloads\new_city.csv", encoding='utf-8-sig')

# 불필요한 컬럼 제거 코드
drop_cols = ['매수자', '매도자', '해제사유발생일', '거래유형', '중개사소재지', '등기일자']
df = df.drop(columns=drop_cols)

print(f"컬럼 제거 후: {df.columns.tolist()}")
print(f"데이터 건수: {len(df)}")

df.to_csv(r"C:\Users\2471369\Downloads\new_city2.csv", index=False, encoding='utf-8-sig')
print("저장 완료")

컬럼 제거 후: ['도시명', '시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일', '거래금액(만원)', '동', '층', '건축년도', '도로명', '자족용지비율', '단지별_세대수', '도시별_세대수']
데이터 건수: 50307
저장 완료
